# Retrieval Augmented Generation
## Step 3: Tokens and Tokenization
The first step is to load our model onto the GPU for faster inference. Note that we load the model and tokenizer separately and keep them as such so that we can explore them separately.
- Note: Model and Tokenizer are from the same LLM
- We use LLM from HuggingFace to demonstrate the Tokenization process

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

/users/tuev/llm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:44<00:00, 22.21s/it]


In [8]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|>"

# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
print("Input IDs: ", input_ids)


Input IDs:  tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001]], device='cuda:0')


In [17]:
for id in input_ids[0]:
    print(tokenizer.decode([id]), end=" ")

Write an email apolog izing to Sarah for the trag ic garden ing m ish ap . Exp lain how it happened . <|assistant|> 

In [13]:
# Generate the text
generation_output = model.generate(
  input_ids=input_ids,
  max_new_tokens=200
)
print(generation_output)


tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001,  3323,   622, 29901,   317,  3742,   406,
          6225, 11763,   363,   278, 19906,   292,   341,   728,   481,    13,
            13,    13, 29928,   799, 19235, 29892,    13,    13,    13, 29902,
          4966,   445,  2643, 14061,   366,  1532, 29889,   306,   626,  5007,
           304,  4653,   590,  6483,   342,  3095, 11763,   363,   278,   443,
          6477,   403, 15134,   393, 10761,   297,   596, 16423, 22600, 29889,
            13,    13,    13,  2887,   366,  1073, 29892,   306,   505,  2337,
          7336,  2859,   278, 15409,   322, 22024,   339,  1793,   310,   596,
         16423, 29889,   739,   471,   411,  2107, 23451,   358,   393,   306,
         16277,   287,   278, 11423,   284, 18658,  8581,   304,   596,  1339,
          8238, 11492, 27089,   267, 29889,   306,  

In [14]:
# Print the output
print(tokenizer.decode(generation_output[0]))

Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|> Subject: Sincere Apologies for the Gardening Mishap


Dear Sarah,


I hope this message finds you well. I am writing to express my deepest apologies for the unfortunate incident that occurred in your garden yesterday.


As you know, I have always admired the beauty and tranquility of your garden. It was with great disappointment that I witnessed the accidental damage caused to your beloved rose bushes. I understand how much effort and care you put into maintaining your garden, and it pains me to have caused any harm to it.


The incident happened when I was attempting to trim the overgrown hedges. Unfortunately, in my haste, I misjudged the distance and accidentally clipped a few of your rose bushes. I realize now that I should have been more cautious and respectful of your property.


Please know that


In [16]:
print(tokenizer.decode(3323))
print(tokenizer.decode(622))
print(tokenizer.decode([3323, 622]))
print(tokenizer.decode(29901))

Sub
ject
Subject
:


# Comparing Trained LLM Tokenizers


In [18]:
import html

colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]


def show_tokens(sentence, tokenizer_name, style="auto"):
    """Highlight tokens. In notebooks, HTML works in Colab, VS Code/Cursor, and Jupyter; ANSI is for plain terminals.

    style: "auto" (HTML in IPython, else ANSI), "html", or "ansi".
    """
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids


    from IPython.display import HTML, display

    parts = []
    for idx, t in enumerate(token_ids):
        piece = tokenizer.decode([t])
        rgb = colors_list[idx % len(colors_list)].replace(";", ",")
        esc = html.escape(piece, quote=True)
        parts.append(
            "<span style="
            f'"background-color: rgb({rgb}); color: #1a1a1a; margin-right: 4px; '
            f'padding: 1px 4px; border-radius: 3px; display: inline-block;">'
            f"{esc}</span>"
        )
    display(HTML(" ".join(parts)))
    return


In [19]:
text = """
English and CAPITALIZATION
🎵 鸟
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600
"""

In [20]:
show_tokens(text, "bert-base-uncased")

In [21]:
show_tokens(text, "bert-base-cased")

In [22]:
show_tokens(text, "gpt2")

In [23]:
show_tokens(text, "google/flan-t5-small")

In [24]:
# The official is `tiktoken` but this the same tokenizer on the HF platform
show_tokens(text, "Xenova/gpt-4")

In [25]:
# You need to request access before being able to use this tokenizer
show_tokens(text, "bigcode/starcoder2-15b")

In [26]:
show_tokens(text, "facebook/galactica-1.3b")

In [27]:
show_tokens(text, "microsoft/Phi-3-mini-4k-instruct")